# 101 Spark basics

The goal of this lab is to get familiar with Spark programming.

- Scala
    - [Spark programming guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
    - [RDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/RDD.html)
    - [PairRDD APIs](https://spark.apache.org/docs/latest/api/scala/org/apache/spark/rdd/PairRDDFunctions.html)
- Python
    - [Spark programming guide](https://spark.apache.org/docs/3.5.0/rdd-programming-guide.html)
    - [All RDD APIs](https://spark.apache.org/docs/3.5.0/api/python/reference/api/pyspark.RDD.html)

Use `Tab` for autocompletion, `Shift+Tab` for documentation.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local") \
    .appName("Local Spark") \
    .config('spark.ui.port', '4040') \
    .getOrCreate()
sc = spark.sparkContext

sc

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


<SparkContext master=local appName=Local Spark>

## 101-1 Spark warm-up

Load the ```capra``` and ```divinacommedia``` datasets and try the following actions:
- Show their content (```collect```)
- Count their rows (```count```)
- Split phrases into words (```map``` or ```flatMap```; what’s the difference?)
- Check the results (remember: evaluation is lazy)
- Try the ```toDebugString``` function to check the execution plan
    - In PySpark, use ```toDebugString().decode("unicode_escape")```

In [3]:
rddCapra = sc.textFile("../../../../datasets/capra.txt")
rddDC = sc.textFile("../../../../datasets/divinacommedia.txt")
print("First RDD contains:", rddCapra.collect())
print("The number of rows of the first RDD is: ", rddCapra.count())
print("Splitting into words: \n using 'map': ", rddCapra.map(lambda x: x.split(" ")).collect(), \
      "\n using flatMap: ", rddCapra.flatMap(lambda x: x.split(" ")).collect())
rddL = rddCapra. \
   flatMap(lambda x : x.split(" ") ). \
   map(lambda x : (x,1)). \
   reduceByKey(lambda x,y : x+y)
print(rddL.toDebugString().decode("unicode_escape"))


First RDD contains: ['sopra la panca la capra campa', 'sotto la panca la capra crepa']
The number of rows of the first RDD is:  2
Splitting into words: 
 using 'map':  [['sopra', 'la', 'panca', 'la', 'capra', 'campa'], ['sotto', 'la', 'panca', 'la', 'capra', 'crepa']] 
 using flatMap:  ['sopra', 'la', 'panca', 'la', 'capra', 'campa', 'sotto', 'la', 'panca', 'la', 'capra', 'crepa']
(1) PythonRDD[11] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[10] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[9] at partitionBy at DirectMethodHandleAccessor.java:104 []
 +-(1) PairwiseRDD[8] at reduceByKey at /tmp/ipykernel_883/3398705532.py:10 []
    |  PythonRDD[7] at reduceByKey at /tmp/ipykernel_883/3398705532.py:10 []
    |  ../../../../datasets/capra.txt MapPartitionsRDD[1] at textFile at DirectMethodHandleAccessor.java:104 []
    |  ../../../../datasets/capra.txt HadoopRDD[0] at textFile at DirectMethodHandleAccessor.java:104 []


## 101-2 Basic Spark jobs

Implement on Spark the following jobs and test them on both capra and divinacommedia datasets.

- **Word count**: count the number of occurrences of each word
  - Result: (sopra, 1), (la, 4), …
- **Word length count**: count the number of occurrences of words of given lengths
  - Result: (2, 4), (5, 8)
- Count the average length of words given their first letter (i.e., words that begin with "s" have an average length of 5)
  - Result: (s, 5), (l, 2), …
- Return the inverted index of words (i.e., for each word, list the numbers of lines in which they appear)
  - Result: (sopra, (0)), (la, (0, 1)), ...

Also, check how sorting works and try to sort key-value RDDs by descending values.

In [25]:
rddCW = rddCapra.flatMap(lambda x: x.split(" "))
rddDCW = rddDC.flatMap(lambda x: x.split(" "))

#ES 1
print("Word count:", \
      rddCW.map(lambda x: (x, 1)).\
        reduceByKey(lambda v1, v2: v1 + v2). \
        map(lambda kv: (kv[1], kv[0])). \
        sortByKey(True). \
            collect())

#ES 2
print("Word length count(rddCapra):", \
      rddCW.map(lambda x: (len(x), 1)). \
        reduceByKey(lambda v1, v2: v1 + v2). \
        collect())
print("Word length count(rddDC):", \
      rddDCW.map(lambda x: (len(x), 1)). \
        reduceByKey(lambda v1, v2: v1 + v2). \
        collect())

#ES 3
print("Average length(rddCapra):", \
      rddCW.map(lambda x: (x[0].lower(), (len(x), 1))). \
        reduceByKey(lambda v1, v2: (v1[0] + v2[0], v1[1] + v2[1])). \
        mapValues(lambda v: v[0] / v[1]). \
        # map(lambda x: (x[0], x[1][0] / x[1][1])). \
        collect())

print("Average length(rddDC):", \
      rddDCW.filter(lambda x: len(x) > 0). \
        map(lambda x: (x[0].lower(), (len(x), 1))). \
        reduceByKey(lambda v1, v2: (v1[0] + v2[0], v1[1] + v2[1])). \
        mapValues(lambda v: v[0] / v[1]). \
        collect())

#ES 4
print("Inverted Index:", \
      rddCW.zipWithIndex(). \
        groupByKey(). \
        mapValues(lambda x: list(x)). \
        collect())

# print("Inverted Index:", \
#       rddDCW.zipWithIndex(). \
#         groupByKey(). \
#         mapValues(lambda x: list(x)). \
#         collect())

Word count: [(1, 'sopra'), (1, 'campa'), (1, 'sotto'), (1, 'crepa'), (2, 'panca'), (2, 'capra'), (4, 'la')]
Word length count(rddCapra): [(5, 8), (2, 4)]
Word length count(rddDC): [(2, 19258), (6, 11775), (8, 5363), (5, 13504), (9, 3231), (7, 7379), (0, 10754), (1, 6992), (3, 16887), (4, 9111), (11, 933), (10, 1741), (15, 18), (12, 370), (14, 50), (13, 154), (16, 3), (17, 1)]
Average length(rddCapra): [('s', 5.0), ('l', 2.0), ('p', 5.0), ('c', 5.0)]
Average length(rddDC): [('l', 3.7876783849634528), ('d', 4.28529945553539), ('c', 4.7628874592697805), ('a', 3.9546289493019837), ('i', 3.411481900452489), ('n', 3.729512587038029), ('m', 4.565703971119134), ('v', 5.298060809193201), ('r', 7.038872691933917), ('p', 5.133871380988983), ('u', 3.8171641791044775), ('s', 4.8543489449305195), ('o', 4.3238095238095235), ('e', 1.854820415879017), ('q', 5.50783378746594), ('�', 1.0769230769230769), ('f', 4.983743967487935), ('t', 4.7542118624509575), ('b', 5.8014925373134325), ('g', 5.6815542652970

## 101-3 Extra Spark jobs

Implement the following job.

- Co-occurrence count: count the number of co-occurrences in the text. A co-occurrence is defined as "two distinct words appearing in the same line".
  - In the first line of the *capra* dataset, co-occurrences are:
     - (sopra, la), (sopra, panca), (sopra, capra), (sopra, campa)
     - (la, sopra), (la, panca), (la, capra), (la, campa) 
     - (panca, sopra), (panca, la), (panca, capra), (panca, campa)
     - (capra, sopra), (capra, la), (capra, panca), (capra, campa)
     - (campa, sopra), (campa, la), (campa, panca), (campa, capra)